# Ep_ISA_NEW rerun notebook: preflight audit -> ISA -> checks

This notebook reruns the updated Ep_ISA_NEW workflow. It does not draw Nature figures.

Order:
1. Install and import Ep_ISA_NEW.
2. Load models, promoter regions and FASTA.
3. Build Fi-NeMo motif_locs for CAGE, DEV and HK.
4. Run preflight motif single/pair and overlap audit first.
5. Rerun ISA with updated deepISA-style interaction logic.
6. Check null counts, pair counts, NaN interaction counts and normalized interaction behavior.

## 0. Runtime setup

Before running this in Colab, put the `Ep_ISA_NEW` folder somewhere in Google Drive, or upload `Ep_ISA_NEW.zip` to `/content`. Then set `EP_ISA_NEW_DIR` below if needed.

In [ ]:
!pip install tensorflow tf-keras bioframe pyBigWig loguru statsmodels h5py matplotlib-venn -q

import os, sys, json, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from loguru import logger

from google.colab import drive
drive.mount('/content/drive')

# EDIT THIS if your Drive location differs.
EP_ISA_NEW_DIR = Path('/content/drive/MyDrive/DeepEpromote/Drosophila/ISA/Ep_ISA_NEW')

# Optional fallback: upload /content/Ep_ISA_NEW.zip and this cell will unpack it.
if not EP_ISA_NEW_DIR.exists() and Path('/content/Ep_ISA_NEW.zip').exists():
    with zipfile.ZipFile('/content/Ep_ISA_NEW.zip') as zf:
        zf.extractall('/content')
    EP_ISA_NEW_DIR = Path('/content/Ep_ISA_NEW')

assert EP_ISA_NEW_DIR.exists(), f'Ep_ISA_NEW_DIR not found: {EP_ISA_NEW_DIR}'
!pip install -e "$EP_ISA_NEW_DIR" -q

from Ep_ISA_NEW.quickstart import EpQuickStart
from Ep_ISA_NEW.scoring.preflight import run_preflight_pair_audit
print('Using Ep_ISA_NEW from:', EP_ISA_NEW_DIR)

## 1. Paths and configuration

In [ ]:
BASE_DIR    = Path('/content/drive/MyDrive/DeepEpromote/Drosophila')
IC_TRIM_DIR = BASE_DIR / 'Motif_cluster/ic_trimmed_results'
SCAN_DIR    = IC_TRIM_DIR / 'finemo_validation_scans/stage2_core_promoter/finemo_scans'
FINEMO_H5   = IC_TRIM_DIR / 'IC_Trimmed_MetaClusters_for_Finemo.h5'
PROM_FILE   = BASE_DIR / 'DeepCAGE/DATA/starr_cage_reconstructed.tsv'
CAGE_MODEL  = BASE_DIR / 'DeepCAGE/model/model_starr_ctss_T1/best_model.h5'
STARR_JSON  = BASE_DIR / 'DeepSTARR/model_artifacts/DeepSTARR.model.json'
STARR_H5    = BASE_DIR / 'DeepSTARR/model_artifacts/DeepSTARR.model.h5'

# New output root. This keeps old results_cage/results_dev/results_hk untouched.
OUT_ROOT = IC_TRIM_DIR / 'ep_isa_new_rerun_results'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

TASKS = {
    'CAGE': {'scan': 'CAGE_NEW', 'results': OUT_ROOT / 'results_cage_newisa', 'track': 0, 'model': 'cage'},
    'DEV':  {'scan': 'DEV',      'results': OUT_ROOT / 'results_dev_newisa',  'track': 0, 'model': 'starr'},
    'HK':   {'scan': 'HK',       'results': OUT_ROOT / 'results_hk_newisa',   'track': 1, 'model': 'starr'},
}

ISA_CONFIG_BASE = {
    'null_percentile': 80,
    'min_count': 10,
    'q_val_thresh': 0.1,
    'receptive_field': 255,
    'single_null_n_samples': 8192,
    'pair_null_n_samples': 8192,
    'pair_null_n_bins': 20,
    'tau_quantile': 50.0,
    'num_regions_per_batch': 200,
    'pred_batch_size': 1024,
}

for p in [SCAN_DIR, FINEMO_H5, PROM_FILE, CAGE_MODEL, STARR_JSON, STARR_H5]:
    assert Path(p).exists(), f'Missing path: {p}'
print('Output root:', OUT_ROOT)

## 2. Load models

In [ ]:
cage_model = tf.keras.models.load_model(
    str(CAGE_MODEL), custom_objects={'mse': tf.keras.losses.MeanSquaredError()})
print(f'CAGE model: input={cage_model.input_shape}, output={cage_model.output_shape}')

import tf_keras
with open(STARR_JSON, 'r') as f:
    src = tf_keras.models.model_from_json(f.read())
src.load_weights(str(STARR_H5))
tmp = '/content/_tmp_starr_ep_isa_new.h5'
src.save(tmp)
starr_model = tf.keras.models.load_model(tmp, compile=False)
os.remove(tmp)
print(f'DeepSTARR model: input={starr_model.input_shape}, output={starr_model.output_shape}  [0]=DEV [1]=HK')

## 3. Build promoter regions and FASTA

In [ ]:
prom = pd.read_csv(PROM_FILE, sep='\t').rename(columns={
    'Chromosome': 'chrom', 'Start': 'start', 'End': 'end',
    'CAGE_dom_strand': 'strand', 'CAGE_dom_log2TPM': 'y', 'Sequence': 'seq'})
prom['sc'] = 'mismatch'
prom.loc[prom['refseq_tss_strand'] == prom['strand'], 'sc'] = 'match'
prom.loc[prom[['refseq_tss_strand', 'strand']].isna().any(axis=1), 'sc'] = 'NA'
prom_19777 = prom[prom.sc == 'match'].reset_index(drop=True)
assert len(prom_19777) == 19777

df_regions = prom_19777[['chrom', 'start', 'end']].copy()
df_regions['peak_id'] = df_regions.index
df_regions = df_regions[['peak_id', 'chrom', 'start', 'end']]

FASTA_PATH = Path('/content/prom_regions_dm3_ep_isa_new.fa')
with open(FASTA_PATH, 'w') as f:
    for _, r in prom_19777.iterrows():
        f.write(f">{r['chrom']}:{int(r['start'])}-{int(r['end'])}\n{r['seq']}\n")

print('regions:', len(df_regions))
print('FASTA:', FASTA_PATH)

# Sanity check: Ep_ISA_NEW supports this region-level FASTA directly.
import bioframe as bf
from Ep_ISA_NEW.scoring.utils_isa import region_str_to_seq
fasta_check = bf.load_fasta(str(FASTA_PATH))
seq_lens = [len(region_str_to_seq(fasta_check, f"{r.chrom}:{int(r.start)}-{int(r.end)}")) for _, r in df_regions.head(5).iterrows()]
print('First 5 FASTA lengths via Ep_ISA_NEW:', seq_lens)
assert len(set(seq_lens)) == 1, seq_lens

## 4. Create task objects and Fi-NeMo motif_locs

This step writes `motif_locs.csv` and `non_motif_locs.csv`, but does not run ISA yet.

In [ ]:
MODEL_BY_NAME = {'cage': cage_model, 'starr': starr_model}
qs = {}

for task, cfg in TASKS.items():
    results_dir = Path(cfg['results'])
    results_dir.mkdir(parents=True, exist_ok=True)
    q = EpQuickStart(results_dir=str(results_dir), fasta_path=str(FASTA_PATH), df_regions=df_regions)
    q.define_model(MODEL_BY_NAME[cfg['model']])
    q.load_finemo(
        hits_tsv_path=str(SCAN_DIR / cfg['scan'] / 'hits.tsv'),
        finemo_h5_path=str(FINEMO_H5),
        auto_threshold_percentile=50,
    )
    qs[task] = q
    print(task, 'motif_locs ->', q.files['motif_locs'])

## 5. Preflight audit first

Run this before any ISA scoring. Check whether single motif loci already exceed valid non-overlapping motif pairs, and inspect skipped overlap/abutting pairs.

In [ ]:
preflight_rows = []
for task, q in qs.items():
    cfg = TASKS[task]
    summary = run_preflight_pair_audit(
        motif_locs_path=q.files['motif_locs'],
        out_summary_path=q.files['preflight_pair_audit'],
        out_region_path=q.files['preflight_pair_audit_by_region'],
        out_overlap_path=q.files['preflight_overlap_pairs'],
        receptive_field=ISA_CONFIG_BASE['receptive_field'],
    )
    row = summary.iloc[0].to_dict()
    row['task'] = task
    preflight_rows.append(row)

preflight = pd.DataFrame(preflight_rows).set_index('task')
display(preflight[[
    'motif_locs_rows_raw', 'motif_locs_rows_after_new_dedup',
    'all_pairs_same_region', 'receptive_field_pairs',
    'pair_to_single_ratio_receptive_field',
    'overlapping_or_adjacent_pairs', 'overlap_exact_same_tf_pairs',
    'overlap_different_family_pairs', 'too_far_pairs'
]])

In [ ]:
# Top skipped overlap/abutting TF pairs. These are not standard ISA motif pairs.
for task, q in qs.items():
    p = Path(q.files['preflight_overlap_pairs'])
    if not p.exists() or p.stat().st_size == 0:
        print(task, 'no overlap/abutting pairs')
        continue
    df = pd.read_csv(p)
    if df.empty:
        print(task, 'no overlap/abutting pairs')
        continue
    pair = df.apply(lambda r: '|'.join(sorted([str(r.tf1), str(r.tf2)])), axis=1)
    print('\n' + task)
    display(pair.value_counts().head(15).rename('n').to_frame())

## 5b. Count ledger: raw motifs -> candidate pairs -> ISA outputs

This table is the main count check. Before full ISA, it shows raw Fi-NeMo hits, motif_locs after Ep_ISA_NEW loading/thresholding/deduplication, and candidate motif pairs under the deepISA pair rule. After full ISA, rerun this cell with `include_algorithm_outputs=True` to add `motif_single_isa.csv` and `motif_combi_isa.csv` counts.

In [ ]:
def build_count_ledger(include_algorithm_outputs=False):
    rows = []
    for task, q in qs.items():
        cfg = TASKS[task]
        scan_hits = SCAN_DIR / cfg['scan'] / 'hits.tsv'
        motif_locs = Path(q.files['motif_locs'])
        preflight_path = Path(q.files['preflight_pair_audit'])

        raw_hits_n = len(pd.read_csv(scan_hits, sep='\t'))
        motif_locs_n = len(pd.read_csv(motif_locs)) if motif_locs.exists() else np.nan
        pf = pd.read_csv(preflight_path).iloc[0] if preflight_path.exists() else None

        row = {
            'task': task,
            'raw_finemo_hits_tsv': raw_hits_n,
            'motif_locs_after_load_finemo': motif_locs_n,
            'motif_locs_after_new_dedup': int(pf.motif_locs_rows_after_new_dedup) if pf is not None else np.nan,
            'all_same_region_candidate_pairs': int(pf.all_pairs_same_region) if pf is not None else np.nan,
            'valid_nonoverlap_rf_candidate_pairs': int(pf.receptive_field_pairs) if pf is not None else np.nan,
            'skipped_overlap_or_abutting_pairs': int(pf.overlapping_or_adjacent_pairs) if pf is not None else np.nan,
            'skipped_too_far_pairs': int(pf.too_far_pairs) if pf is not None else np.nan,
        }

        if include_algorithm_outputs:
            data = Path(q.data_dir)
            single_path = data / 'motif_single_isa.csv'
            combi_path = data / 'motif_combi_isa.csv'
            row['single_motif_after_single_isa_filter'] = len(pd.read_csv(single_path)) if single_path.exists() else np.nan
            row['motif_pairs_after_combi_isa'] = len(pd.read_csv(combi_path)) if combi_path.exists() else np.nan
            if combi_path.exists():
                combi = pd.read_csv(combi_path)
                inter = f"interaction_t{cfg['track']}"
                row['motif_pairs_with_non_nan_new_interaction'] = int(combi[inter].notna().sum()) if inter in combi.columns else np.nan
                row['motif_pairs_with_nan_new_interaction'] = int(combi[inter].isna().sum()) if inter in combi.columns else np.nan

        rows.append(row)
    return pd.DataFrame(rows)

count_ledger_pre_isa = build_count_ledger(include_algorithm_outputs=False)
display(count_ledger_pre_isa)
count_ledger_pre_isa.to_csv(OUT_ROOT / 'ep_isa_new_count_ledger_pre_isa.csv', index=False)

## 6. Rerun ISA with Ep_ISA_NEW

This is the long model-scoring step. It writes updated single ISA, combi ISA, null tables, normalized interaction and coop summaries.

Set `RUN_FULL_ISA = True` when the preflight audit looks acceptable.

In [ ]:
RUN_FULL_ISA = False  # change to True to start the full rerun

if not RUN_FULL_ISA:
    print('Full ISA rerun is disabled. Set RUN_FULL_ISA=True after checking preflight tables.')
else:
    for task, q in qs.items():
        cfg = dict(ISA_CONFIG_BASE)
        cfg['tracks'] = [TASKS[task]['track']]
        print('\n=== Running', task, '===')
        q.run_isa(isa_config=cfg, start_from='single_isa')
        print(task, 'done ->', q.results_dir)

## 7. Post-run checks

Run after the full ISA rerun. Checks are designed for updated deepISA-style normalized interaction.

In [ ]:
count_ledger_post_isa = build_count_ledger(include_algorithm_outputs=True)
display(count_ledger_post_isa)

post_count_path = OUT_ROOT / 'ep_isa_new_count_ledger_post_isa.csv'
count_ledger_post_isa.to_csv(post_count_path, index=False)
print('Saved:', post_count_path)

In [ ]:
def task_check(task, q, track):
    data = Path(q.data_dir)
    required = ['motif_single_isa.csv', 'motif_combi_isa.csv', 'null_isa.csv', 'null_interaction.csv']
    missing = [x for x in required if not (data / x).exists()]
    if missing:
        return {'task': task, 'status': 'missing', 'missing': ';'.join(missing)}

    single = pd.read_csv(data / 'motif_single_isa.csv')
    combi = pd.read_csv(data / 'motif_combi_isa.csv')
    null_isa = pd.read_csv(data / 'null_isa.csv')
    null_inter = pd.read_csv(data / 'null_interaction.csv')
    inter = f'interaction_t{track}'
    raw_formula = combi[f'isa1_t{track}'] + combi[f'isa2_t{track}'] - combi[f'isa_both_t{track}']
    nonnan = combi[inter].notna()
    median_abs_diff_from_raw = float((combi.loc[nonnan, inter] - raw_formula.loc[nonnan]).abs().median()) if nonnan.any() else np.nan

    return {
        'task': task,
        'status': 'ok',
        'single_rows': len(single),
        'combi_rows': len(combi),
        'null_isa_rows': len(null_isa),
        'null_interaction_rows': len(null_inter),
        'interaction_non_nan_rows': int(nonnan.sum()),
        'interaction_nan_rows': int((~nonnan).sum()),
        'interaction_nan_fraction': float((~nonnan).mean()),
        'median_abs_diff_interaction_vs_raw_formula': median_abs_diff_from_raw,
        'interaction_min': float(combi[inter].min(skipna=True)),
        'interaction_max': float(combi[inter].max(skipna=True)),
    }

checks = []
for task, q in qs.items():
    checks.append(task_check(task, q, TASKS[task]['track']))
checks = pd.DataFrame(checks)
display(checks)

check_path = OUT_ROOT / 'ep_isa_new_postrun_checks.csv'
checks.to_csv(check_path, index=False)
print('Saved:', check_path)

## 8. Output handoff to plotting

Use these result folders as input for a new plotting notebook only after checks pass:

- CAGE: `results_cage_newisa`
- DEV: `results_dev_newisa`
- HK: `results_hk_newisa`

Do not mix these updated ISA outputs with the old `results_cage/results_dev/results_hk` outputs.